# ED Fig 3o — FOXF1 versus BMP4 pixel density in one cyst

**Feeds:** **ED Fig 3o**, drawn by the second-to-last cell; the last cell writes the table in
`derived/`. `scripts/render_ed3o_panel.py` redraws the same panel from that table, with no images.

Ported from `BMP-FOXF1 correlation-20260918.ipynb`, by T.H., which draws the panel from one z plane
of the cyst shown in ED Fig 3n. The recipe below is unchanged: the same DAPI threshold, the same
background offsets, the same 99.9th-percentile outlier clip, the same 256 x 256 histogram and the
same least-squares fit.

**Changes from the original notebook**
1. The acquisition is read from `data/` under this lane, or from `ED3O_DATA_DIR`, instead of the
   notebook's own `E:\` paths, and a missing file now says so rather than failing inside `tifffile`.
2. The last cell writes a CSV to `output/derived/` so a re-run cannot overwrite the shipped table.
   The original wrote an `.xlsx` to its own output directory; the columns and rounding are the same.
3. The panel is saved under `output/` for the same reason, rather than to the original's `OUT` path.
4. Stored outputs were cleared; they carried the original machine's file paths.

**This notebook supersedes the port of `BMP-FOXF1 correlation-20260916.ipynb`.** That version read
three separately cropped single-plane TIFFs at 558 x 362 and dropped bright specks with bounds set by
hand, at 1200 for FOXF1 and 6000 for BMP4. This one reads the whole 583 x 388 acquisition, crops
nothing, and replaces those bounds with a percentile computed from the data. The pixel count changes
from 94,476 to 95,287 and R² from 0.640 to 0.646.

**Not in this repository:** the acquisition `BMP4-FOXF1-z8.tif`. Raw imaging is available from the
lead contact on request. Everything the panel needs is in `derived/`.

In [ ]:
import os
import numpy as np
import pandas as pd
import tifffile
import matplotlib.pyplot as plt
from matplotlib import colors
import scipy.ndimage
from scipy.stats import linregress


## Configuration


In [ ]:
import os

# The acquisition lives in this lane's data/ directory unless ED3O_DATA_DIR says otherwise.
DATA = os.environ.get("ED3O_DATA_DIR",
                      os.path.join(os.path.dirname(os.getcwd()), "data"))
# Written beside the notebook, not over the shipped table in derived/.
OUT = os.path.join(os.path.dirname(os.getcwd()), "output", "derived")
os.makedirs(OUT, exist_ok=True)

IMAGE    = "BMP4-FOXF1-z8.tif"   # confocal acquisition, ZCYX = (14, 5, 388, 583), 16-bit

Z_INDEX  = 5                     # sixth z-plane, 0-based index
C_DAPI   = 1                     # channel 2
C_BMP4   = 3                     # channel 4
C_FOXF1  = 4                     # channel 5

# Nuclear mask. Only pixels with DAPI above this level are analysed, so the plot
# describes signal within tissue rather than in the surrounding medium.
DAPI_THRESHOLD = 1000

# Background offsets, read off the intensity histograms plotted below, in raw 16-bit
# grey levels. Each channel has its own offset because the background level differs
# between channels.
BG_FOXF1, BG_BMP4 = 1724, 880

# Outlier clip. The brightest 100 - CLIP_PERCENTILE per cent of pixels in each channel
# are dropped, so that a handful of very bright specks cannot drive the fit. The same
# rule is applied to both channels and the cut-offs it produces are printed below,
# rather than being set by hand.
CLIP_PERCENTILE = 99.9

HIST_BINS      = 256
CONTOUR_LEVELS = np.linspace(0.25, 1, 4)
SMOOTH_SIGMA   = (2, 1)

PDF_NAME  = "ED3o_BMP4_vs_FOXF1.pdf"


## Load the three channels


In [ ]:
path = os.path.join(DATA, IMAGE)
if not os.path.isfile(path):
    raise SystemExit(
        f"{IMAGE} is not in {DATA}. This notebook reads the acquisition of the cyst in "
        f"ED Fig 3n, which is not in this repository; the panel itself is redrawn from "
        f"derived/ by scripts/render_ed3o_panel.py.")

stack = tifffile.imread(path)                               # (Z, C, Y, X)
print("file  :", IMAGE, stack.shape, stack.dtype)

# The whole acquired field is analysed; nothing is cropped away.
dapi  = stack[Z_INDEX, C_DAPI]
bmp4  = stack[Z_INDEX, C_BMP4]
foxf1 = stack[Z_INDEX, C_FOXF1]
print("plane : z = %d of %d,  %d x %d px" % (Z_INDEX + 1, stack.shape[0], *dapi.shape))


## Background levels

The offsets set above were chosen from these histograms. Each channel has a large
background peak, and the offset sits at its upper edge; the vertical line marks the
value used.


In [ ]:
plt.figure(figsize=(10, 4))
for i, (img, name, bg) in enumerate(((foxf1, "FOXF1", BG_FOXF1),
                                     (bmp4,  "BMP4",  BG_BMP4))):
    plt.subplot(1, 2, i + 1)
    plt.hist(img.ravel(), bins=100, color="C%d" % i, alpha=0.7)
    plt.axvline(bg, color="k", lw=1)
    plt.title("%s intensity histogram" % name)
    plt.xlabel("Intensity")
    plt.ylim(0, 10000)
plt.subplot(1, 2, 1); plt.ylabel("Pixel count")
plt.tight_layout()
plt.show()


## Nuclear mask, background subtraction and outlier clip

Three steps, in order: keep pixels inside the nucleus, subtract the background of each
channel and keep those still above it, then drop the brightest tail of each channel.
The count removed at every step is printed, so none of them is doing unnoticed work.


In [ ]:
nuclear = dapi > DAPI_THRESHOLD

x = foxf1[nuclear].astype(np.int64) - BG_FOXF1     # x axis: FOXF1
y = bmp4[nuclear].astype(np.int64) - BG_BMP4       # y axis: BMP4

print("nuclear mask (DAPI > %d)       : %6d px" % (DAPI_THRESHOLD, nuclear.sum()))
print("  FOXF1 at or below background : %6d" % (x <= 0).sum())
print("  BMP4  at or below background : %6d" % (y <= 0).sum())

above = (x > 0) & (y > 0)
x, y = x[above], y[above]
print("above background in both       : %6d" % x.size)

# Cut-offs come from this population, so they are a property of the data rather than
# a chosen number.
cut_foxf1 = np.percentile(x, CLIP_PERCENTILE)
cut_bmp4  = np.percentile(y, CLIP_PERCENTILE)
print("  %.1f-th percentile, FOXF1    : %6.0f" % (CLIP_PERCENTILE, cut_foxf1))
print("  %.1f-th percentile, BMP4     : %6.0f" % (CLIP_PERCENTILE, cut_bmp4))

keep = (x <= cut_foxf1) & (y <= cut_bmp4)
print("  removed by the clip          : %6d" % (~keep).sum())
x, y = x[keep], y[keep]
print("retained                       : %6d" % x.size)


## Normalization

Each axis is scaled to span 0 to 1, so the plot shows relative rather than absolute
intensity. The correlation below is computed on the background-subtracted values and
is unchanged by this step.


In [ ]:
x_norm = (x - x.min()) / (x.max() - x.min())
y_norm = (y - y.min()) / (y.max() - y.min())


## Pixel-density plot

Pixels are binned on a 256 x 256 grid. The colour shows `log(1 + count)` scaled to its
maximum, so that the sparse high-intensity tail stays visible next to the dense
low-intensity corner. Contours are drawn on a lightly smoothed copy of the same grid.


In [ ]:
H, xedges, yedges = np.histogram2d(x_norm, y_norm, bins=HIST_BINS)
H_log  = np.log1p(H)
H_norm = H_log / H_log.max()
H_smooth = scipy.ndimage.gaussian_filter(H_norm.astype(float), sigma=SMOOTH_SIGMA)

white_to_green = colors.LinearSegmentedColormap.from_list(
    "WhiteToGreen", ["#FFFFFF", "#B5E48C", "#76C893", "#1A7431"])

plt.style.use("default")
fig, ax = plt.subplots(facecolor="white")
im = ax.imshow(H_norm.T, origin="lower", cmap=white_to_green,
               extent=[xedges[0], xedges[-1], yedges[0], yedges[-1]],
               aspect="auto", vmin=0, vmax=1)
ax.set_aspect("equal")
ax.contour(0.5 * (xedges[:-1] + xedges[1:]),
           0.5 * (yedges[:-1] + yedges[1:]),
           H_smooth.T, levels=CONTOUR_LEVELS, colors="black", linewidths=0.5)

ax.set_xlabel("FOXF1 normalized intensity")
ax.set_ylabel("BMP4 normalized intensity")
ax.set_title("BMP4 versus FOXF1 pixel density")
fig.colorbar(im, ax=ax, label="Pixel density")
plt.tight_layout()

plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"]  = 42
plt.savefig(os.path.join(os.path.dirname(OUT), PDF_NAME), format="pdf", dpi=300)
plt.show()
print("saved:", PDF_NAME)


## Correlation


In [ ]:
fit = linregress(x, y)
r_squared = fit.rvalue ** 2
print("n  = %d pixels" % x.size)
print("r  = %.4f" % fit.rvalue)
print("R2 = %.9f" % r_squared)


## Export

One row per analysed pixel, holding the two plotted coordinates. The layout matches
the one used for Extended Data Fig. 3g, so the sheet can be dropped straight into the
source-data workbook.


In [ ]:
out = pd.DataFrame({
    "FOXF1 normalized intensity": x_norm.round(6),
    "BMP4 normalized intensity":  y_norm.round(6),
})
out.to_csv(os.path.join(OUT, "ed3o_per_pixel.csv"), index=False)
print("%d rows written   R2 = %.9f" % (len(out), r_squared))
